In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully! ✅")


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/processed/manali_places_enriched.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 20)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,nature,history,culture,adventure,photography,shopping,religious,family,travel_tags,interest_count
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,1,1,1,0,1,1,1,0,"culture, history, nature, photography, religio...",6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,1,1,1,0,1,0,0,1,"culture, family, history, nature, photography",5
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,0,1,0,0,1,0,0,0,"history, photography",2
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,0,0,0,0,0,0,0,0,NaN,0
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,1,1,0,0,1,0,0,1,"family, history, nature, photography",4


In [3]:
feature_columns = [
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]

X_structured = (
    df[feature_columns]
    .fillna(0)
    .astype(float)
)

print("Structured feature matrix:", X_structured.shape)


Structured feature matrix: (20, 8)


In [4]:
user_preferences = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.2,
    "adventure": 0.4,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.2
}

user_vector = np.array([
    user_preferences[feature]
    for feature in feature_columns
]).reshape(1, -1)

print("User vector:", user_vector)


User vector: [[1.  0.  0.2 0.4 1.  0.  0.  0.2]]


In [5]:
structured_scores = cosine_similarity(
    user_vector,
    X_structured
).flatten()

df["structured_score"] = structured_scores

df[[
    "name",
    "structured_score"
]].sort_values(
    "structured_score",
    ascending=False
).head(10)


,name,structured_score
5,Van Vihar National Park,0.734968
4,Jogini Falls,0.734968
1,Old Manali snow point,0.717137
6,Manali View Point,0.668153
8,Lama Dugh Trek Start Point,0.668153
0,Hadimba Devi Temple,0.600099
10,Kharma valley,0.472456
2,Nehru Kund,0.472456
14,Baror Parsha Waterfall,0.472456
18,Gulaba Viewpoint,0.462910


In [6]:
df["recommendation_text"] = (
    df["name"].fillna("") + " "
    + df["category"].fillna("") + " "
    + df["travel_tags"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["recommendation_text"].astype(str)
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (20, 122)


In [7]:
query = "quiet scenic places for taking pictures"

query_vector = tfidf_vectorizer.transform([query])

tfidf_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

df["tfidf_score"] = tfidf_scores

df[[
    "name",
    "travel_tags",
    "tfidf_score"
]].sort_values(
    "tfidf_score",
    ascending=False
).head(10)


,name,travel_tags,tfidf_score
0,Hadimba Devi Temple,"culture, history, nature, photography, religio...",0.0
1,Old Manali snow point,"culture, family, history, nature, photography",0.0
2,Nehru Kund,"history, photography",0.0
3,Kullu Manali River rafting,NaN,0.0
4,Jogini Falls,"family, history, nature, photography",0.0
5,Van Vihar National Park,"family, history, nature, photography",0.0
6,Manali View Point,photography,0.0
7,Rahala Waterfalls,"family, history, nature",0.0
8,Lama Dugh Trek Start Point,nature,0.0
9,Atal Bihari statue,history,0.0


## 8. Load the semantic model

We use the same model as notebook 06.


In [8]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Semantic model loaded! ✅")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5242.63it/s]


Semantic model loaded! ✅


In [9]:
df["semantic_text"] = (
    df["name"].fillna("") + ". "
    + df["category"].fillna("") + ". "
    + df["travel_tags"].fillna("")
)

place_embeddings = model.encode(
    df["semantic_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", place_embeddings.shape)


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.14it/s]

Embedding matrix shape: (20, 384)


In [10]:
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

semantic_scores = cosine_similarity(
    query_embedding,
    place_embeddings
).flatten()

df["semantic_score"] = semantic_scores

df[[
    "name",
    "travel_tags",
    "semantic_score"
]].sort_values(
    "semantic_score",
    ascending=False
).head(10)


,name,travel_tags,semantic_score
6,Manali View Point,photography,0.535230
5,Van Vihar National Park,"family, history, nature, photography",0.447241
4,Jogini Falls,"family, history, nature, photography",0.418287
2,Nehru Kund,"history, photography",0.417762
17,Hadimba Forest Block,NaN,0.378226
18,Gulaba Viewpoint,"family, history, nature",0.376310
1,Old Manali snow point,"culture, family, history, nature, photography",0.374756
10,Kharma valley,"history, nature",0.361882
15,Old Manali View point,"family, history",0.344807
14,Baror Parsha Waterfall,"history, nature",0.339228


In [11]:
# Rating normalization
rating_min = df["rating"].min()
rating_max = df["rating"].max()

if rating_max == rating_min:
    df["rating_score"] = 1.0
else:
    df["rating_score"] = (
        (df["rating"] - rating_min)
        / (rating_max - rating_min)
    )

# Popularity normalization
df["log_reviews"] = np.log1p(
    df["reviews"].clip(lower=0)
)

popularity_min = df["log_reviews"].min()
popularity_max = df["log_reviews"].max()

if popularity_max == popularity_min:
    df["popularity_score"] = 1.0
else:
    df["popularity_score"] = (
        (df["log_reviews"] - popularity_min)
        / (popularity_max - popularity_min)
    )

df[[
    "name",
    "rating",
    "rating_score",
    "reviews",
    "popularity_score"
]].head(10)


,name,rating,rating_score,reviews,popularity_score
0,Hadimba Devi Temple,4.6,0.777778,49688,1.000000
1,Old Manali snow point,4.6,0.777778,428,0.429428
2,Nehru Kund,4.4,0.555556,7767,0.777182
3,Kullu Manali River rafting,4.5,0.666667,88,0.240583
4,Jogini Falls,4.6,0.777778,10842,0.817225
5,Van Vihar National Park,4.2,0.333333,9050,0.795536
6,Manali View Point,4.6,0.777778,87,0.239227
7,Rahala Waterfalls,4.5,0.666667,797,0.503949
8,Lama Dugh Trek Start Point,4.6,0.777778,297,0.385680
9,Atal Bihari statue,4.5,0.666667,74,0.220034


## 12. Define hybrid weights

These are **baseline weights**.

```text
25% Structured personalization
20% TF-IDF
30% Semantic relevance
15% Rating
10% Popularity
```

The semantic model gets the largest single weight because natural-language understanding is central to our application.

Later, we can tune or learn these weights from feedback data.


In [12]:
weights = {
    "structured": 0.25,
    "tfidf": 0.20,
    "semantic": 0.30,
    "rating": 0.15,
    "popularity": 0.10
}

print("Total weight:", sum(weights.values()))


Total weight: 1.0


In [13]:
df["hybrid_score"] = (
    weights["structured"] * df["structured_score"]
    + weights["tfidf"] * df["tfidf_score"]
    + weights["semantic"] * df["semantic_score"]
    + weights["rating"] * df["rating_score"]
    + weights["popularity"] * df["popularity_score"]
)

hybrid_columns = [
    "name",
    "rating",
    "reviews",
    "travel_tags",
    "structured_score",
    "tfidf_score",
    "semantic_score",
    "rating_score",
    "popularity_score",
    "hybrid_score"
]

df[hybrid_columns].sort_values(
    "hybrid_score",
    ascending=False
).head(10)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
4,Jogini Falls,4.6,10842,"family, history, nature, photography",0.734968,0.0,0.418287,0.777778,0.817225,0.507617
6,Manali View Point,4.6,87,photography,0.668153,0.0,0.535230,0.777778,0.239227,0.468197
0,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.600099,0.0,0.321686,0.777778,1.000000,0.463197
1,Old Manali snow point,4.6,428,"culture, family, history, nature, photography",0.717137,0.0,0.374756,0.777778,0.429428,0.451321
5,Van Vihar National Park,4.2,9050,"family, history, nature, photography",0.734968,0.0,0.447241,0.333333,0.795536,0.447468
8,Lama Dugh Trek Start Point,4.6,297,nature,0.668153,0.0,0.338459,0.777778,0.385680,0.423811
10,Kharma valley,4.8,143,"history, nature",0.472456,0.0,0.361882,1.000000,0.298357,0.406514
2,Nehru Kund,4.4,7767,"history, photography",0.472456,0.0,0.417762,0.555556,0.777182,0.404494
14,Baror Parsha Waterfall,4.7,489,"history, nature",0.472456,0.0,0.339228,0.888889,0.445391,0.397755
18,Gulaba Viewpoint,4.5,3576,"family, history, nature",0.462910,0.0,0.376310,0.666667,0.684071,0.397028


In [14]:
def recommend_hybrid(
    query,
    user_preferences,
    top_n=5,
    weights=None
):
    if weights is None:
        weights = {
            "structured": 0.25,
            "tfidf": 0.20,
            "semantic": 0.30,
            "rating": 0.15,
            "popularity": 0.10
        }

    # Validate preference keys
    missing = [
        feature
        for feature in feature_columns
        if feature not in user_preferences
    ]

    if missing:
        raise ValueError(
            f"Missing preference features: {missing}"
        )

    # Structured similarity
    user_vector = np.array([
        float(user_preferences[feature])
        for feature in feature_columns
    ]).reshape(1, -1)

    structured_scores = cosine_similarity(
        user_vector,
        X_structured
    ).flatten()

    # TF-IDF similarity
    query_vector = tfidf_vectorizer.transform(
        [query]
    )

    tfidf_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Semantic similarity
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        place_embeddings
    ).flatten()

    # Combine
    result = df.copy()

    result["structured_score"] = structured_scores
    result["tfidf_score"] = tfidf_scores
    result["semantic_score"] = semantic_scores

    result["hybrid_score"] = (
        weights["structured"] * result["structured_score"]
        + weights["tfidf"] * result["tfidf_score"]
        + weights["semantic"] * result["semantic_score"]
        + weights["rating"] * result["rating_score"]
        + weights["popularity"] * result["popularity_score"]
    )

    result = result.sort_values(
        "hybrid_score",
        ascending=False
    )

    return result[[
        "name",
        "rating",
        "reviews",
        "travel_tags",
        "structured_score",
        "tfidf_score",
        "semantic_score",
        "rating_score",
        "popularity_score",
        "hybrid_score"
    ]].head(top_n).reset_index(drop=True)


In [15]:
user_a = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 0.2,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.0
}

recommend_hybrid(
    "peaceful scenic places for taking beautiful photos",
    user_a,
    top_n=5
)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Jogini Falls,4.6,10842,"family, history, nature, photography",0.700140,0.0,0.490182,0.777778,0.817225,0.520479
1,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.571662,0.0,0.449838,0.777778,1.000000,0.494533
2,Manali View Point,4.6,87,photography,0.700140,0.0,0.557698,0.777778,0.239227,0.482934
3,Van Vihar National Park,4.2,9050,"family, history, nature, photography",0.700140,0.0,0.511589,0.333333,0.795536,0.458065
4,Lama Dugh Trek Start Point,4.6,297,nature,0.700140,0.0,0.403852,0.777778,0.385680,0.451425


In [16]:
user_b = {
    "nature": 0.2,
    "history": 1.0,
    "culture": 1.0,
    "adventure": 0.0,
    "photography": 0.4,
    "shopping": 0.0,
    "religious": 0.5,
    "family": 0.2
}

recommend_hybrid(
    "I want to explore historical and cultural places",
    user_b,
    top_n=5
)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.802022,0.0,0.539683,0.777778,1.000000,0.579077
1,Old Manali snow point,4.6,428,"culture, family, history, nature, photography",0.793548,0.0,0.488372,0.777778,0.429428,0.504508
2,Jogini Falls,4.6,10842,"family, history, nature, photography",0.570352,0.0,0.470187,0.777778,0.817225,0.482033
3,Kharma valley,4.8,143,"history, nature",0.537733,0.0,0.542310,1.000000,0.298357,0.476962
4,Shiv Mahadev Temple,4.6,270,"history, religious",0.672166,0.0,0.497741,0.777778,0.374277,0.471458


In [17]:
user_c = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 1.0,
    "photography": 0.5,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.0
}

recommend_hybrid(
    "exciting outdoor activities, trekking and mountain adventures",
    user_c,
    top_n=5
)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Lama Dugh Trek Start Point,4.6,297,nature,0.666667,0.0,0.518987,0.777778,0.385680,0.477597
1,Jogini Falls,4.6,10842,"family, history, nature, photography",0.500000,0.0,0.362650,0.777778,0.817225,0.432184
2,Kharma valley,4.8,143,"history, nature",0.471405,0.0,0.389941,1.000000,0.298357,0.414669
3,Baror Parsha Waterfall,4.7,489,"history, nature",0.471405,0.0,0.386826,0.888889,0.445391,0.411771
4,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.408248,0.0,0.245071,0.777778,1.000000,0.392250


## 18. Test User D: Family Trip 👨‍👩‍👧

In [18]:
user_d = {
    "nature": 0.7,
    "history": 0.2,
    "culture": 0.3,
    "adventure": 0.2,
    "photography": 0.4,
    "shopping": 0.2,
    "religious": 0.1,
    "family": 1.0
}

recommend_hybrid(
    "family friendly outdoor places and enjoyable attractions",
    user_d,
    top_n=5
)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Jogini Falls,4.6,10842,"family, history, nature, photography",0.840963,0.210359,0.493916,0.777778,0.817225,0.598877
1,Gulaba Viewpoint,4.5,3576,"family, history, nature",0.802181,0.226049,0.448601,0.666667,0.684071,0.548742
2,Old Manali snow point,4.6,428,"culture, family, history, nature, photography",0.850291,0.165737,0.445717,0.777778,0.429428,0.539045
3,Rahala Waterfalls,4.5,797,"family, history, nature",0.802181,0.226049,0.421273,0.666667,0.503949,0.522532
4,Van Vihar National Park,4.2,9050,"family, history, nature, photography",0.840963,0.167216,0.468804,0.333333,0.795536,0.513879


In [19]:
example_recommendations = recommend_hybrid(
    "peaceful scenic places for taking beautiful photos",
    user_a,
    top_n=3
)

example_recommendations


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Jogini Falls,4.6,10842,"family, history, nature, photography",0.700140,0.0,0.490182,0.777778,0.817225,0.520479
1,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.571662,0.0,0.449838,0.777778,1.000000,0.494533
2,Manali View Point,4.6,87,photography,0.700140,0.0,0.557698,0.777778,0.239227,0.482934


In [20]:
semantic_heavy_weights = {
    "structured": 0.15,
    "tfidf": 0.15,
    "semantic": 0.50,
    "rating": 0.10,
    "popularity": 0.10
}

recommend_hybrid(
    "peaceful scenic places for taking beautiful photos",
    user_a,
    top_n=5,
    weights=semantic_heavy_weights
)


,name,rating,reviews,travel_tags,structured_score,tfidf_score,semantic_score,rating_score,popularity_score,hybrid_score
0,Jogini Falls,4.6,10842,"family, history, nature, photography",0.700140,0.0,0.490182,0.777778,0.817225,0.509612
1,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.571662,0.0,0.449838,0.777778,1.000000,0.488446
2,Manali View Point,4.6,87,photography,0.700140,0.0,0.557698,0.777778,0.239227,0.485571
3,Van Vihar National Park,4.2,9050,"family, history, nature, photography",0.700140,0.0,0.511589,0.333333,0.795536,0.473703
4,Nehru Kund,4.4,7767,"history, photography",0.495074,0.0,0.482024,0.555556,0.777182,0.448547


## 21. Save hybrid recommendation data

In [21]:
output_path = "../data/processed/manali_hybrid_scores.csv"

df.to_csv(
    output_path,
    index=False
)

embedding_dir = Path("../data/embeddings")
embedding_dir.mkdir(parents=True, exist_ok=True)

np.save(
    embedding_dir / "manali_place_embeddings.npy",
    place_embeddings
)

print("✅ Hybrid scores saved")
print("✅ Embeddings saved")


✅ Hybrid scores saved
✅ Embeddings saved
